In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Google Drive mounted!")

Mounted at /content/drive
Google Drive mounted!


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

INPUT_FILE   = "/content/drive/MyDrive/insider_threat/data/user_features.csv"
SCORES_FILE  = "/content/drive/MyDrive/insider_threat/data/anomaly_scores.csv"
SUMMARY_FILE = "/content/drive/MyDrive/insider_threat/data/model_summary.txt"

print("Libraries loaded!")
print("Setup complete!")

Libraries loaded!
Setup complete!


In [3]:
print("Loading feature table...")

df = pd.read_csv(INPUT_FILE)
print(f"Loaded: {df.shape[0]} users x {df.shape[1]-1} features")
print("Done!")

Loading feature table...
Loaded: 500 users x 26 features
Done!


In [4]:
print("Selecting model features...")

FEATURE_COLS = [
    "total_logins",
    "off_hours_login_rate",
    "weekend_login_rate",
    "unique_pcs_used",
    "avg_login_hour",
    "total_file_events",
    "sensitive_file_rate",
    "file_deletions",
    "file_copies",
    "off_hours_file_rate",
    "total_emails",
    "external_email_rate",
    "attachment_rate",
    "avg_email_size",
    "off_hours_email_rate",
    "usb_connections",
    "off_hours_usb_rate",
]

FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
X_raw = df[FEATURE_COLS].values

print(f"Using {len(FEATURE_COLS)} features")
print("Done!")

Selecting model features...
Using 17 features
Done!


In [5]:
print("Scaling features...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print("Features scaled to mean=0, std=1")
print("Done!")

Scaling features...
Features scaled to mean=0, std=1
Done!


In [6]:
print("Training Isolation Forest model...")

CONTAMINATION = 0.05
N_ESTIMATORS  = 200

model = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=42,
    n_jobs=-1
)
model.fit(X_scaled)

raw_scores       = model.decision_function(X_scaled)
anomaly_scores   = -raw_scores
predictions      = model.predict(X_scaled)

score_min        = anomaly_scores.min()
score_max        = anomaly_scores.max()
risk_score_0_100 = ((anomaly_scores - score_min) / (score_max - score_min)) * 100

print(f"Model trained: {N_ESTIMATORS} trees, contamination={CONTAMINATION}")
print(f"Users flagged as anomalies: {(predictions == -1).sum()}")
print("Done!")

Training Isolation Forest model...
Model trained: 200 trees, contamination=0.05
Users flagged as anomalies: 25
Done!


In [7]:
print("Building results table...")

results = df[["user"] + FEATURE_COLS].copy()
results["anomaly_score_raw"] = anomaly_scores
results["risk_score"]        = risk_score_0_100.round(2)
results["is_anomaly"]        = predictions == -1
results["risk_level"]        = pd.cut(
    results["risk_score"],
    bins=[0, 40, 70, 100],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

results = results.sort_values("risk_score", ascending=False).reset_index(drop=True)
results["rank"] = results.index + 1

results.to_csv(SCORES_FILE, index=False)
print("anomaly_scores.csv saved to Google Drive!")
print("Done!")

Building results table...
anomaly_scores.csv saved to Google Drive!
Done!


In [8]:
print("=" * 55)
print("  TOP 20 HIGHEST RISK USERS")
print("=" * 55)

top20 = results.head(20)[["rank", "user", "risk_score", "risk_level",
                           "total_logins", "off_hours_login_rate",
                           "external_email_rate", "usb_connections"]]
print(top20.to_string(index=False))

  TOP 20 HIGHEST RISK USERS
 rank  user  risk_score risk_level  total_logins  off_hours_login_rate  external_email_rate  usb_connections
    1 U0457      100.00       High           208              0.653846             0.449033             27.0
    2 U0303       99.47       High           183              0.704918             0.466915             39.0
    3 U0013       97.06       High           212              0.646226             0.449189             26.0
    4 U0053       95.26       High           182              0.538462             0.459280             33.0
    5 U0347       94.79       High           175              0.554286             0.465302             34.0
    6 U0115       94.27       High           185              0.605405             0.451721             28.0
    7 U0126       93.54       High           190              0.584211             0.418944             33.0
    8 U0141       93.52       High           196              0.607143             0.448372         

In [10]:
print("=" * 55)
print("  RISK LEVEL DISTRIBUTION")
print("=" * 55)

risk_summary = results["risk_level"].value_counts().sort_index()
for level, count in risk_summary.items():
    pct = count / len(results) * 100
    bar = "█" * int(pct / 2)
    print(f"  {level:6s} : {count:4d} users ({pct:5.1f}%)  {bar}")

  RISK LEVEL DISTRIBUTION
  Low    :  484 users ( 96.8%)  ████████████████████████████████████████████████
  Medium :    1 users (  0.2%)  
  High   :   15 users (  3.0%)  █


In [11]:
summary_lines = [
    "INSIDER THREAT DETECTION — MODEL SUMMARY",
    "=" * 50,
    f"Algorithm         : Isolation Forest",
    f"Number of trees   : {N_ESTIMATORS}",
    f"Contamination rate: {CONTAMINATION}",
    f"Total users scored: {len(results)}",
    f"Anomalies flagged : {results['is_anomaly'].sum()}",
    "",
    "RISK LEVEL BREAKDOWN:",
    risk_summary.to_string(),
    "",
    "TOP 10 HIGHEST RISK USERS:",
    results.head(10)[["rank","user","risk_score","risk_level"]].to_string(index=False),
]

with open(SUMMARY_FILE, "w") as f:
    f.write("\n".join(summary_lines))

print("=" * 55)
print("  MODEL TRAINING COMPLETE")
print("=" * 55)
print(f"  Total users scored : {len(results)}")
print(f"  Anomalies flagged  : {results['is_anomaly'].sum()}")
print("=" * 55)
print("All files saved to Google Drive!")
print("Next step: Run 05_visualize notebook!")
results.head()

  MODEL TRAINING COMPLETE
  Total users scored : 500
  Anomalies flagged  : 25
All files saved to Google Drive!
Next step: Run 05_visualize notebook!


,user,total_logins,off_hours_login_rate,weekend_login_rate,unique_pcs_used,avg_login_hour,total_file_events,sensitive_file_rate,file_deletions,file_copies,...,attachment_rate,avg_email_size,off_hours_email_rate,usb_connections,off_hours_usb_rate,anomaly_score_raw,risk_score,is_anomaly,risk_level,rank
0,U0457,208,0.653846,0.384615,122,11.649038,1365,0.712088,361,323,...,0.613357,1.555820e+06,0.501757,27.0,0.740741,0.253981,100.00,True,High,1
1,U0303,183,0.704918,0.398907,111,10.792350,1391,0.706686,339,376,...,0.587139,1.553063e+06,0.509786,39.0,0.641026,0.252019,99.47,True,High,2
2,U0013,212,0.646226,0.405660,110,12.132075,1481,0.725186,373,360,...,0.600342,1.461026e+06,0.498719,26.0,0.653846,0.243022,97.06,True,High,3
3,U0053,182,0.538462,0.318681,107,12.230769,1353,0.717664,344,316,...,0.569129,1.434227e+06,0.491477,33.0,0.727273,0.236345,95.26,True,High,4
4,U0347,175,0.554286,0.262857,109,11.725714,1454,0.720083,382,391,...,0.598754,1.457611e+06,0.471530,34.0,0.705882,0.234573,94.79,True,High,5
